# Yuki VTuber - Servidor de Voz (GPT-SoVITS)

**Execute as celulas em ordem.**
- Celula 1: Verifica GPU
- Celula 2: Monta Google Drive (salva tudo la, nao perde ao reiniciar)
- Celula 3: Instala GPT-SoVITS
- Celula 4: Baixa modelos (so na primeira vez)
- Celula 5: Upload do audio (so na primeira vez)
- Celula 6: Inicia servidor
- Celula 7: Cria tunel publico (pega a URL aqui)
- Celula 8: Mantem sessao ativa

In [ ]:
# Celula 1 - Verificar GPU
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ERRO: sem GPU - ative em Ambiente de execucao > Alterar tipo')

In [ ]:
# Celula 2 - Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os

# Pasta base no Drive - tudo fica salvo aqui
DRIVE_DIR = '/content/drive/MyDrive/YukiVTuber'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/ref_audio', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)

print(f'Drive montado! Pasta: {DRIVE_DIR}')

In [ ]:
# Celula 3 - Instalar GPT-SoVITS
import os

if not os.path.exists('/content/GPT-SoVITS'):
    !git clone https://github.com/RVC-Boss/GPT-SoVITS /content/GPT-SoVITS

os.chdir('/content/GPT-SoVITS')
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn
print('Instalacao concluida!')

In [ ]:
# Celula 4 - Baixar modelos (usa cache do Drive se ja existir)
import os, shutil

DRIVE_DIR = '/content/drive/MyDrive/YukiVTuber'
LOCAL_MODELS = '/content/GPT-SoVITS/GPT_SoVITS/pretrained_models'
os.makedirs(LOCAL_MODELS, exist_ok=True)

DRIVE_MODELS = f'{DRIVE_DIR}/models'

def get_model(filename, drive_subdir, url):
    local_path = f'{LOCAL_MODELS}/{drive_subdir}/{filename}'
    drive_path = f'{DRIVE_MODELS}/{drive_subdir}/{filename}'
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    if os.path.exists(drive_path):
        print(f'Cache: {filename}')
        if not os.path.exists(local_path):
            shutil.copy(drive_path, local_path)
    else:
        print(f'Baixando {filename}...')
        os.system(f'wget -q -O "{local_path}" "{url}"')
        shutil.copy(local_path, drive_path)
        print(f'Salvo no Drive!')

get_model('s2G2333k.pth', 'gsv-v2final-pretrained',
    'https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v2final-pretrained/s2G2333k.pth')
get_model('s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt', 'gsv-v2final-pretrained',
    'https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch%3D12-step%3D369668.ckpt')

# HuBERT e BERT (clona se nao existir no Drive)
for m in ['chinese-roberta-wwm-ext-large', 'chinese-hubert-base']:
    drive_m = f'{DRIVE_MODELS}/{m}'
    local_m = f'{LOCAL_MODELS}/{m}'
    if os.path.exists(drive_m):
        print(f'Cache: {m}')
        if not os.path.exists(local_m):
            os.symlink(drive_m, local_m)
    else:
        print(f'Clonando {m}...')
        os.system(f'git clone --depth 1 https://huggingface.co/hfl/{m} {drive_m}')
        os.symlink(drive_m, local_m)

print('\nModelos prontos!')

In [ ]:
# Celula 5 - Audio de referencia (usa Drive se ja existir)
import os
from google.colab import files

DRIVE_DIR = '/content/drive/MyDrive/YukiVTuber'
REF_DIR = f'{DRIVE_DIR}/ref_audio'
REF_PATH_FILE = f'{DRIVE_DIR}/ref_audio_path.txt'

# Verificar se ja tem audio salvo no Drive
existing = [f for f in os.listdir(REF_DIR) if f.endswith('.wav')] if os.path.exists(REF_DIR) else []

if existing:
    ref_audio = f'{REF_DIR}/{existing[0]}'
    open(REF_PATH_FILE, 'w').write(ref_audio)
    print(f'Audio ja existe no Drive: {existing[0]}')
    print('Pulando upload!')
else:
    print('Faca upload do arquivo .wav de referencia da Yuki')
    print('Local: C:\\GPT-SoVITS\\...\\output\\slicer_opt\\')
    print('Arquivo: audio [vocals].mp3_0000000000_0000183040.wav')
    uploaded = files.upload()
    for fn, data in uploaded.items():
        dest = f'{REF_DIR}/{fn}'
        open(dest, 'wb').write(data)
        open(REF_PATH_FILE, 'w').write(dest)
        print(f'Salvo no Drive: {dest}')

In [ ]:
# Celula 6 - Iniciar servidor GPT-SoVITS
import os, subprocess, time
os.chdir('/content/GPT-SoVITS')

DRIVE_DIR = '/content/drive/MyDrive/YukiVTuber'
ref = open(f'{DRIVE_DIR}/ref_audio_path.txt').read().strip()

os.makedirs('GPT_SoVITS/configs', exist_ok=True)
config = f"""default:
  device: cuda
  is_half: true
  version: v2
  t2s_weights_path: GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt
  vits_weights_path: GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s2G2333k.pth
  bert_base_path: GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large
  cnhuhbert_base_path: GPT_SoVITS/pretrained_models/chinese-hubert-base
  ref_audio_path: {ref}
  prompt_text: Vamos la menino segura essa espada
  prompt_lang: en
  text_lang: en
  text_split_method: cut5
  batch_size: 1
  media_type: wav
  streaming_mode: false
"""
open('GPT_SoVITS/configs/tts_infer.yaml', 'w').write(config)

proc = subprocess.Popen(
    ['python', 'api_v2.py', '-a', '0.0.0.0', '-p', '9880', '-c', 'GPT_SoVITS/configs/tts_infer.yaml'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
print('Iniciando servidor GPT-SoVITS...')
for i in range(120):
    l = proc.stdout.readline()
    if l.strip(): print(l.strip())
    if 'startup complete' in l or 'Uvicorn running' in l:
        print('\nServidor ativo na porta 9880!')
        break
    time.sleep(1)

In [ ]:
# Celula 7 - Criar tunel publico (COPIE A URL DAQUI)
import subprocess, time, re, os
os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')
tp = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:9880'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('Aguardando URL do tunel...')
for i in range(40):
    l = tp.stdout.readline()
    if 'trycloudflare.com' in l:
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', l)
        if m:
            url = m.group(0)
            print(f'\n=== SERVIDOR ATIVO ===')
            print(f'URL: {url}')
            print(f'\nCole no conf.yaml local do projeto:')
            print(f'  gpt_sovits_tts:')
            print(f'    api_url: "{url}/tts"')
            break
    time.sleep(1)

In [ ]:
# Celula 8 - Manter sessao ativa (nao feche!)
import time
print('Sessao ativa - nao feche esta celula')
print('Enquanto isso, cole a URL do tunel no conf.yaml e inicie o projeto local')
while True:
    time.sleep(60)
    print(f'Online {time.strftime("%H:%M:%S")}')